<a href="https://colab.research.google.com/github/santiagomantillaa-prog/Santiago-mantilla-actividad-2-introducci-n-ing-/blob/main/Copia_de_Copia_Ingesti%C3%B3n_de_Datos_en_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#pip install requests pandas openpyx1

In [ ]:
import requests # importacion de la libreria Requests para enviar peticion a la API https://jsonplaceholder.typicode.com/users
import sqlite3 # importacion de la interface para conexion y creacion de bases de datos SQLite
import pandas as pd # importacion de Pandas, libreria para el analisis de datos en Python
#Las librerias son conjunto de archivos de código preescrito

# Definicion de la URL de la API para envio de peticion y recepcion de datos usando protocolo HTTP
# desde la API https://jsonplaceholder.typicode.com/
API_URL = "https://jsonplaceholder.typicode.com/users"
# Una API es una interfaz con protocolo http que ofrece un conjunto de datos a la cual hay que enviar una solicitud para obtener una respuesta


# Creacion de la funcion extraer_datos validandose codigo 200 (exito en la conexion) o codigo 404 (codigo de error en la conexion)
def extraer_datos(url):
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"Error al conectar con la API: {response.status_code}")
#Esta función recibe una URL, hace una petición y devuelve los datos en formato JSON

# Creacion de la base de datos en SQLite "datos_api.db"
# se realiza la conexion a dicha base de datos
# y se crea el "cursor" para "execute" el CREATE TABLE y el INSERT OR REPLACE INTO usuarios
def guardar_en_db(datos):
    conn = sqlite3.connect('datos_api.db')
    cursor = conn.cursor()
    # El cursor permite ejecutar instrucciones SQL dentro de la base de datos

    # Creación de la tabla "usuarios" dentro de la base de datos "datos_api.db"
    # esta contiene los campos id, nombre, usuario y email
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS usuarios (
            id INTEGER PRIMARY KEY,
            nombre TEXT,
            usuario TEXT,
            email TEXT
        )
    ''')
    #TEXT indica que los campos almacenan cadenas de texto.

    # Inserción de datos en la tabla "usuarios"
    for item in datos:
        cursor.execute('''
            INSERT OR REPLACE INTO usuarios (id, nombre, usuario, email)
            VALUES (?, ?, ?, ?)
        ''', (item['id'], item['name'], item['username'], item['email']))
    # Se usa INSERT OR REPLACE para:
     #  Insertar un nuevo registro si el 'id' no existe
     #  Reemplazar el registro existente si ya hay un usuario con el mismo 'id'
    # Los signos ? son marcadores de posición que permiten prevenir inyección SQL
    # y facilitan pasar los valores como tupla en el segundo argumento

    # Aceptación mediante "Commit" el ingreso de datos en la tabla "usuarios"
    conn.commit()
    # Este comando confirma y guarda de forma permanente todos los cambios

    # Cerrado de la conexion a la base de datos "datos_api.db"
    conn.close()
    # Esto libera los recursos utilizados por la conexión

# 3. GENERAR EVIDENCIAS (Pandas y Auditoría)
def generar_evidencias(datos_api):
    # --- Archivo Excel/CSV con Pandas ---
    conn = sqlite3.connect('datos_api.db')
    # Se establece conexión con la base de datos para extraer la información almacenada
    df_db = pd.read_sql_query("SELECT * FROM usuarios", conn)
    # Se ejecuta una consulta SQL y se carga el resultado directamente en un DataFrame de Pandas
    df_db.to_csv('muestra_usuarios.csv', index=False)
    # Se genera un archivo CSV con toda la información de la tabla "usuarios"
    conn.close()
    # Se cierra la conexión con la base de datos una vez realizada la exportación

    # la variable "registros_api" obtiene la cantidad de registros recibidos desde la
    # API con url "https://jsonplaceholder.typicode.com/users"
    # la variable "registros_db" obtiene la cantidad de registros almacenados en la tabla "usuarios"
    registros_api = len(datos_api)
    # La variable "registros_api" obtiene la cantidad total de registros recibidos desde la API
    registros_db = len(df_db)
    # La variable "registros_db" obtiene la cantidad de registros almacenados en la tabla "usuarios"

    # Creación del archivo "auditoría.txt" con el metodo "open" de Python
    # Se especifica encoding='utf-8' para asegurar que los caracteres especiales
    with open('auditoria.txt', 'w', encoding='utf-8') as f:
        f.write("RESUMEN DE AUDITORÍA\n")
        f.write("====================\n")
         # Escribe un encabezado descriptivo para el reporte de auditoría
        f.write(f"Registros extraídos de la API https://jsonplaceholder.typicode.com/: {registros_api}\n")
        # Registro del número de usuarios obtenidos desde la API
        f.write(f"Registros guardados en la DB de SQLite3: {registros_db}\n")
        # Registro del número de usuarios almacenados en la base de datos SQLite


        if registros_api == registros_db:
            f.write("\nESTADO: Operacion Exitosa.\n")
        else:
            f.write("\nESTADO: Operacion Fallida.\n")
    # Validación: se compara la cantidad de datos obtenidos vs guardados

# EJECUCIÓN DEL PROCESO
try:
    print("Iniciando proceso de petición y recepción de datos desde la API https://jsonplaceholder.typicode.com/users...")
    # Mensaje informativo para indicar el inicio de la petición de datos
    datos = extraer_datos(API_URL)
    # Llamada a la función que obtiene los datos desde la API

    print("Creando la base de datos datos_api.db y guardando datos en la tabla usuarios...")
    #Mensaje informativo sobre la creación de la BD y almacenamiento de datos
    guardar_en_db(datos)
    # Se guardan los datos extraídos dentro de la base de datos SQLite

    print("Generando archivos de evidencia .CSV y auditoría.txt...")
    # Mensaje informativo sobre la generación de evidencias
    generar_evidencias(datos)
    # Se generan los archivos de verificación (CSV + auditoría)

    print("¡Proceso completado con éxito!")
    # Mensaje final indicando que todas las etapas fueron exitosas
except Exception as e:
    print(f"Ocurrió un error: {e}")
    # Bloque de captura de errores

Iniciando proceso de petición y recepción de datos desde la API https://jsonplaceholder.typicode.com/users...
Creando la base de datos datos_api.db y guardando datos en la tabla usuarios...
Generando archivos de evidencia .CSV y auditoría.txt...
¡Proceso completado con éxito!
